# 3 · Claude Code II: Debugging, Testing & Financial Analytics

**Outcome of this session:** the difference between code that *runs* and code you can *trust* — and the discipline that turns one into the other. You debug a broken version of Session 2's Apple valuation with Claude's help, protect it with sanity checks that fail loudly, and then build an Earnings Analysis Engine whose output you machine-verify. The deliverable is an AI-generated investment-memo draft, produced by your own engine.

**In this notebook you will:**

- Diagnose three realistic defects with Claude: reproduce, explain, fix, verify
- Write sanity checks for financial calculations: units, magnitudes, tie-outs
- Build an Earnings Analysis Engine that structures a whole earnings call
- Machine-verify every quoted claim, and produce an investment-memo draft

## The debugging checklist

The same five steps, whether the bug is in Python or in a spreadsheet:

1. **Reproduce** it — run the failing thing again and read the symptom.
2. **Isolate** it — which stage produced the bad value?
3. **Explain** it — paste the symptom to Claude and demand the *cause* before any fix.
4. **Fix** one thing.
5. **Verify** — a sanity check against a figure you already know.

In [1]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

repo root: /Users/ignaciocastro/iese-ai-finance-bootcamp-student
API key:   configured


## Part A · Code that runs versus code you can trust

Session 2's tool produced Apple's peer-implied value, and you accepted it because checks passed. Today the question behind that habit: **how do you prove a financial calculation is right?** Four kinds of sanity check do most of the work, and all four appear in this notebook:

- **Order of magnitude** — a profit margin cannot be 119%; a check should say so loudly.
- **Tie-outs** — compare a computed figure with one that is publicly known. Apple's market value is about **$4.5 trillion**; if your code says $4.5 *billion*, the code is wrong.
- **Unit consistency** — millions versus billions versus units is the most common real-world finance bug.
- **Loud failure on bad data** — a merge that silently produces missing values corrupts everything downstream; an assertion should stop it at the source.

The broken script below is Session 2's Apple valuation with **three realistic defects planted** — a wrong formula, a silent data problem, and a unit error. For each defect: run the buggy cell and read its absurd output aloud; run the ✅ check to see it fail loudly; ask Claude to *explain the cause* before fixing; repair one line in the fix cell; and run the check again — green proves the repair.

In [2]:
import numpy as np
import pandas as pd
pd.options.display.float_format = "{:,.1f}".format

TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "CRM", "ORCL"]
DATA = ROOT / "session-02-coding-copilot" / "data"
fundamentals = pd.read_csv(DATA / "tech_financials.csv")
fundamentals = fundamentals[fundamentals["ticker"].isin(TICKERS)].reset_index(drop=True)
fundamentals = fundamentals.drop(columns=["price_usd", "price_asof"])   # prices arrive separately below
print(f"{len(fundamentals)} companies loaded (fundamentals from SEC filings, as in Session 2)")

8 companies loaded (fundamentals from SEC filings, as in Session 2)


### Defect 1 · The wrong formula

A margin divides profit by **revenue**. The bug cell divides by the wrong base — a realistic one-keystroke slip, and the code *runs without any error*.

**Your task, five steps (the checklist in action):**

1. **Run the bug cell** and read the output — the number is impossible, yet nothing crashed.
2. **Run the ✅ check cell** below the fix: it fails loudly and names the problem. (In this notebook the fix cell starts as a commented gap, so the check fails until you repair it.)
3. **Ask Claude for the cause**: highlight the line marked `<- defect`, then press `Option+K` (Mac) / `Alt+K` (Windows). That single keystroke sends the selection with its address — file, cell, exact line numbers — into the ✱ Claude panel as attached context; you will see the reference appear in the panel's input box. Then type *"this fails the sanity check below — explain the cause before proposing a fix"* and press Enter. The strongest version also pastes the check's error message into the same question. (If the shortcut does nothing: copy the line and the error message into the panel by hand — same result.)
4. **Fill the gap** in the fix cell and run it.
5. **Run the ✅ check again** — green proves the repair.

In [3]:
# THE BUG - run me and read the margin column
df = fundamentals.copy()
df["op_margin"] = df["operating_income_m"] / df["net_income_m"]     # <- defect 1 lives here
df[["ticker", "operating_income_m", "net_income_m", "op_margin"]].head(3)

,ticker,operating_income_m,net_income_m,op_margin
0,AAPL,"133,050.0","112,010.0",1.2
1,MSFT,"155,237.0","133,749.0",1.2
2,NVDA,"130,387.0","120,067.0",1.1


In [5]:
# THE FIX - fill the gap, then run me
### START CODE HERE ###
df["op_margin"] = df["operating_income_m"] / df["revenue_m"]    # a margin divides profit by what?
### END CODE HERE ###
# now run the ✅ check below: it must pass.
print("fixed - now run the ✅ check below")

fixed - now run the ✅ check below


In [6]:
# ✅ sanity check: an operating margin is profit per unit of REVENUE - always between -100% and 100%.
bad = df[(df["op_margin"] < -1) | (df["op_margin"] > 1)]
assert bad.empty, f"margin out of range for {list(bad['ticker'])} - a margin above 100% means the base is wrong"
print("All margins within range ✅")

All margins within range ✅


### Defect 2 · The silent data problem

Prices arrive from a second source and are **merged** onto the fundamentals by ticker. One ticker in the price table is dirty (`" aapl "` — spaces and lowercase), so the merge finds no match and produces a **missing value, silently**. Nothing crashes; every later number for Apple would simply be empty.

Same five steps as Defect 1: bug → check fails → ask for the cause → fill the gap → check passes. The fix here normalizes the join key.

In [7]:
# THE BUG - run me and find Apple's price in the table
prices = pd.DataFrame({
    "ticker": [" aapl ", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "CRM", "ORCL"],   # <- defect 2 lives here
    "price_usd": [309.35, 483.24, 214.72, 344.82, 258.63, 549.90, 209.17, 146.47],
})
df = df.merge(prices, on="ticker", how="left")
df[["ticker", "price_usd"]]

,ticker,price_usd
0,AAPL,NaN
1,MSFT,483.2
2,NVDA,214.7
3,GOOGL,344.8
4,AMZN,258.6
5,META,549.9
6,CRM,209.2
7,ORCL,146.5


In [9]:
# THE FIX - fill the gap, then run me
# Undo the bad merge if it ran, then fix the source table and merge again.
# (Run the BUG cell first - it creates the prices table this cell repairs.)
df = df.drop(columns=["price_usd"], errors="ignore")
### START CODE HERE ###
prices["ticker"] = prices["ticker"].str.strip().str.upper()   # which column is the join key to clean?
### END CODE HERE ###
df = df.merge(prices, on="ticker", how="left")
print("fixed - now run the ✅ check below")

fixed - now run the ✅ check below


In [10]:
# ✅ sanity check: after a merge, nothing may be silently missing.
missing = df[df["price_usd"].isna()]
assert missing.empty, f"price missing after merge for {list(missing['ticker'])} - check ticker formatting on BOTH sides"
print("No missing values after the merge ✅")

No missing values after the merge ✅


### Defect 3 · The unit error

`shares_m` is already in **millions**, so `shares_m × price` gives market value in millions of dollars. The bug cell "converts units" that needed no converting, making every market value a thousand times too small.

Same five steps. The check this time is a **tie-out**: Apple's market value is publicly known to be roughly $4.5 trillion.

In [11]:
# THE BUG - run me and read the printed line out loud
df["mcap_m"] = (df["shares_m"] / 1000) * df["price_usd"]     # <- defect 3 lives here
print(f"Apple market value: ${df.set_index('ticker').loc['AAPL','mcap_m']/1e6:,.4f} trillion  <- read this number out loud")

Apple market value: $0.0045 trillion  <- read this number out loud


In [12]:
# THE FIX - fill the gap, then run me
### START CODE HERE ###
df["mcap_m"] = df["shares_m"] * df["price_usd"]    # shares are ALREADY in millions - no conversion
### END CODE HERE ###
print("fixed - now run the ✅ tie-out below")

fixed - now run the ✅ tie-out below


In [13]:
# ✅ sanity check (tie-out): Apple's market value is publicly known to be roughly $4.5 trillion.
aapl_tn = df.set_index("ticker").loc["AAPL", "mcap_m"] / 1e6
assert 3 < aapl_tn < 6, f"Apple computes to ${aapl_tn:,.3f}tn - off by a factor of ~1000 means a unit error"
print(f"Tie-out passed ✅  Apple: ${aapl_tn:,.2f}tn, in line with public figures")

Tie-out passed ✅  Apple: $4.51tn, in line with public figures


### The repaired analysis, end to end

Three defects, three sanity checks, three one-line fixes — each explained by Claude *before* it was fixed, each proven by a check afterwards. The cell below reruns Session 2's headline with the repaired data, as a final tie-out: same peer-median valuation of Apple as Session 2.

In [14]:
df["ev_m"] = df["mcap_m"] + df["total_debt_m"] - df["cash_m"]
df["ebitda_m"] = df["operating_income_m"] + df["d_and_a_m"]
df["ev_ebitda"] = df["ev_m"] / df["ebitda_m"]

target = df.set_index("ticker").loc["AAPL"]
peer_median = df.loc[df["ticker"] != "AAPL", "ev_ebitda"].median()
implied = (peer_median * target["ebitda_m"] - target["total_debt_m"] + target["cash_m"]) / target["shares_m"]
print(f"peer median rating {peer_median:.1f}x  ->  Apple implied ${implied:,.0f}  vs actual ${target['price_usd']}")
print("Same finding as Session 2 - the repaired pipeline reproduces it. That agreement is itself a tie-out.")

peer median rating 19.5x  ->  Apple implied $191  vs actual $309.35
Same finding as Session 2 - the repaired pipeline reproduces it. That agreement is itself a tie-out.


## Part C · Lab 2: find the fabricated quote

Session 2 ended with a finding: the market prices Apple as if it were nearly the best company in its peer group. Judging whether that confidence is deserved takes **evidence** about growth, margins and management's plans — and the richest recurring source of evidence is the **earnings call**, the quarterly conference where management presents results and analysts push back.

An AI can read a whole call and draft the analysis in seconds. Here is the whole engine on one line:

```
transcript → Claude, forced into a schema where EVERY claim carries a verbatim
quote → YOUR code checks each quote against the document → a memo, every claim
marked verified or not
```

And here is today's game: **the analysis you are about to see contains eleven quoted claims, and exactly one of the quotes is fabricated** — fluent, plausible, and appearing nowhere in the transcript. Reading rarely finds it. Your code will.

(The company is fictional for one honest reason: a real company's calls are in the model's training data, so only a fictional one proves the claims come from the *document*, not from memory. The engine runs unchanged on any real transcript.)

In [15]:
sys.path.insert(0, str(ROOT / "session-03-debugging" / "lab"))
import earnings_starter as engine
from earnings_starter import EARNINGS_SCHEMA, analyze, DEFAULT_TRANSCRIPT
from toolkit import llm            # for llm.show(): renders long output readably

# The grounding rules, applied at the source (the same discipline as Session 1):
engine.SYSTEM = """You are a buy-side equity analyst preparing an internal note.
- Use ONLY the transcript provided. No outside knowledge, no memory of other companies.
- Every evidence_quote must be VERBATIM from the transcript; each will be machine-checked.
- If the transcript does not support a claim, do not make it.
- Separate management's framing from fact; note what guidance excludes."""

print(f"Loading the earnings-call transcript from a file bundled with the course:")
print(f"  {DEFAULT_TRANSCRIPT.relative_to(ROOT)}")
transcript = DEFAULT_TRANSCRIPT.read_text()
print(f"  loaded: {len(transcript.split()):,} words | speakers: CEO, CFO, five analysts | 7 planted red flags")
print()
print("How you would obtain a real one: earnings calls are NOT on EDGAR. Companies")
print("post recordings and prepared remarks on their investor-relations pages, and")
print("transcript providers offer them through paid APIs. The engine below runs")
print("unchanged on any transcript you drop into a text file.")

Loading the earnings-call transcript from a file bundled with the course:
  session-03-debugging/data/transcript_meridian_q2_fy2026.txt
  loaded: 2,164 words | speakers: CEO, CFO, five analysts | 7 planted red flags

How you would obtain a real one: earnings calls are NOT on EDGAR. Companies
post recordings and prepared remarks on their investor-relations pages, and
transcript providers offer them through paid APIs. The engine below runs
unchanged on any transcript you drop into a text file.


### First, see what the engine produces

Run the cell below. It loads a **canned demonstration analysis** (the same shape a live call produces — you run it live at the end) and renders it. Read it the way a portfolio manager would: it looks complete, professional, and quotable.

Somewhere in it is the lie.

In [16]:
analysis = analyze(transcript, dry_run=True)   # canned demonstration output; live run at the end
from toolkit import llm as _llm
preview = [f"**Sentiment:** {analysis['overall_sentiment']} — {analysis['sentiment_rationale']}", ""]
for section, key in [("key_themes", "theme"), ("risks", "risk"), ("red_flags", "flag")]:
    preview.append(f"**{section.replace('_', ' ').title()}**")
    for it in analysis[section]:
        preview.append(f"- {it[key]}")
        preview.append(f"  - *\"{it['evidence_quote'][:120]}...\"*" if len(it['evidence_quote']) > 120
                       else f"  - *\"{it['evidence_quote']}\"*")
    preview.append("")
_llm.show("\n".join(preview), title="The engine's analysis - 11 quoted claims, 1 fabricated")

**The engine's analysis - 11 quoted claims, 1 fabricated**

**Sentiment:** cautiously_positive — Record revenue, backlog and hyperscaler wins argue for strength, but margin pressure framed as one-time for a third straight quarter, rising customer concentration, an inventory and DSO build, and an unquantified export-license overhang all temper the print.

**Key Themes**
- Supply-constrained AI accelerator demand
  - *"Demand for our Corvus accelerators continues to outpace our ability to supply"*
- Data center now dominates the mix (71% of revenue, +61% y/y)
  - *"Data center revenue was 1.72 billion dollars this quarter, up 61 percent year over year"*
- Capacity secured through 2027 via foundry agreement
  - *"secures a doubling of advanced-node wafer allocation through calendar 2027"*
- Margin recovery promised
  - *"we expect gross margins to return to 62 percent by year end"*

**Risks**
- Export license review could remove up to a low-teens share of revenue
  - *"our guidance excludes any impact from the export license review announced in June"*
- Customer concentration rising sharply
  - *"our largest customer represented 22 percent of revenue in the quarter, up from 15 percent a year ago"*
- Inventory build ahead of an unproven product ramp
  - *"Inventory ended the quarter at 1.98 billion dollars, up 41 percent year over year"*
- Working-capital quality deteriorating as payment terms lengthen
  - *"Days sales outstanding were 71 days, compared with 58 a year ago"*

**Red Flags**
- Recurring 'one-time' ramp costs
  - *"this is the third consecutive quarter you've described ramp costs as one-time"*
- CEO/CFO margin-message gap
  - *"we don't guide gross margin beyond the next quarter"*
- Unfalsifiable demand claim
  - *"AI demand is effectively infinite over any horizon we can plan for"*


### Exercise 3: build the detector

Could you spot the fabricated quote by reading? Most people cannot — it is written in the transcript's own style. The comparison is always **the AI's quote against the very document it claims to quote** — a citation check, in code. The detector is simple and merciless: for every claim, take its quote, **normalize** both the quote and the transcript (lowercase, collapse whitespace, straighten curly quotes — so formatting cannot cause false alarms), and test whether the quote actually appears (Python's plain `quote in transcript` substring test — normalization is what makes that dumb test reliable). Mark each claim `verified` or not.

Two gaps: what to normalize as the haystack, and what to look for in it.

In [17]:
import re

def _normalize(text: str) -> str:
    """Whitespace-collapse + casefold + straighten curly quotes. GIVEN - it's
    plumbing; YOUR work is the verification logic below."""
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", text).casefold().strip()

def verify_evidence(analysis: dict, transcript: str) -> dict:
### START CODE HERE ###
    haystack = _normalize(None)                        # normalize which text?
    checked = failed = 0
    for section in ("key_themes", "risks", "red_flags"):
        for item in analysis.get(section, []):
            quote = item.get("evidence_quote", "")
            item["verified"] = bool(quote) and None in haystack   # hint: the NORMALIZED quote
            checked += 1
            failed += 0 if item["verified"] else 1
    analysis["_verification"] = {"quotes_checked": None, "quotes_failed": None}
### END CODE HERE ###
    return analysis

print("defined - now catch a fabrication:")

defined - now catch a fabrication:


In [22]:
import re

def _normalize(text: str) -> str:
    """Whitespace-collapse + casefold + straighten curly quotes. GIVEN - it's
    plumbing; YOUR work is the verification logic below."""
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", text).casefold().strip()

def verify_evidence(analysis: dict, transcript: str) -> dict:
### START CODE HERE ###
    haystack = _normalize(transcript)                        # normalize which text?
    checked = failed = 0
    for section in ("key_themes", "risks", "red_flags"):
        for item in analysis.get(section, []):
            quote = item.get("evidence_quote", "")
            item["verified"] = bool(quote) and _normalize(quote) in haystack   # hint: the NORMALIZED quote
            checked += 1
            failed += 0 if item["verified"] else 1
    analysis["_verification"] = {"quotes_checked": checked, "quotes_failed": failed}
### END CODE HERE ###
    return analysis

print("defined - now catch a fabrication:")

defined - now catch a fabrication:


**One professional aside — the same gate, in the industry's words.** The engine's schema is a raw dictionary; professional Python declares such contracts with **Pydantic** (one class per object, one typed field per key). The cell below re-validates the analysis through typed models: same *shape* gate, but malformed output now raises a precise `ValidationError` naming the offending field, and downstream code reads `analysis.red_flags[0].flag` instead of string keys — which is exactly what the memo function below does: it accepts the *typed* object and uses attribute access throughout.

Note the division of labor: **Pydantic checks the container; `verify_evidence` checks the contents.** A perfectly-shaped analysis can still hold an invented quote — structure and truth are separate gates, and professional pipelines run both.

In [19]:
from pydantic import BaseModel

class Theme(BaseModel):
    theme: str
    evidence_quote: str
    verified: bool | None = None

class Risk(BaseModel):
    risk: str
    severity: str
    evidence_quote: str
    verified: bool | None = None

class RedFlag(BaseModel):
    flag: str
    why_it_matters: str
    evidence_quote: str
    verified: bool | None = None

class EarningsAnalysis(BaseModel):
    overall_sentiment: str
    sentiment_rationale: str
    key_themes: list[Theme]
    risks: list[Risk]
    red_flags: list[RedFlag]

typed = EarningsAnalysis.model_validate(analysis)   # raises ValidationError on any shape violation
print(f"Validated: {len(typed.key_themes)} themes, {len(typed.risks)} risks, "
      f"{len(typed.red_flags)} red flags - all typed.")
print("First red flag, by attribute access:", typed.red_flags[0].flag)

Validated: 4 themes, 4 risks, 3 red flags - all typed.
First red flag, by attribute access: Recurring 'one-time' ramp costs


In [20]:
# See the gate reject, twice - run me. (We break a COPY; the real analysis is untouched.)
import json
from pydantic import ValidationError

broken = json.loads(json.dumps(analysis))
del broken["risks"][1]["evidence_quote"]          # a claim with no evidence
try:
    EarningsAnalysis.model_validate(broken)
except ValidationError as e:
    print("missing evidence  ->", str(e).splitlines()[1].strip(), "| Field required")

broken2 = json.loads(json.dumps(analysis))
broken2["key_themes"][0]["theme"] = 42            # a number where text belongs
try:
    EarningsAnalysis.model_validate(broken2)
except ValidationError as e:
    print("wrong type        ->", str(e).splitlines()[1].strip(), "| should be a string")

print()
print("And the honest limit: the analysis containing the FABRICATED quote passes this")
print("gate completely - shape is not truth. Only your verify_evidence catches the lie.")

missing evidence  -> risks.1.evidence_quote | Field required
wrong type        -> key_themes.0.theme | should be a string

And the honest limit: the analysis containing the FABRICATED quote passes this
gate completely - shape is not truth. Only your verify_evidence catches the lie.


**The reveal.** The caught quote — a promised margin recovery — is entirely plausible and appears **nowhere** in the transcript. Confirm it physically: copy a distinctive phrase from the caught quote, open `session-03-debugging/data/transcript_meridian_q2_fy2026.txt`, and search for it (`Cmd+F` / `Ctrl+F`). Nothing. Reading would likely have missed it; your ten lines of Python did not.

**What to expect on the live run below.** There the model writes its own quotes, and a current model quotes accurately most of the time, so expect **all or nearly all** to verify. If one or two are flagged, that is the checker doing its job: the model paraphrased instead of quoting verbatim — read the flagged lines and judge them yourself. In production the value of this layer is not that it fires often; it is that nothing reaches a committee unchecked.

In [23]:
def memo_from(a: EarningsAnalysis, source_name: str) -> str:
    """A compact analyst note from the TYPED analysis: every claim printed with its
    verification mark. Note the attribute access throughout - this is what the
    Pydantic gate bought us: typed fields instead of string keys."""
    lines = [f"# Meridian Semiconductor - Q2 FY2026 note (draft, from {source_name})",
             f"\n**Sentiment:** {a.overall_sentiment} - {a.sentiment_rationale}\n"]
    checked = failed = 0
    for section_name, items, label in [("Key themes", a.key_themes, "theme"),
                                       ("Risks", a.risks, "risk"),
                                       ("Red flags", a.red_flags, "flag")]:
        lines.append(f"## {section_name}\n")
        for it in items:
            checked += 1
            failed += 0 if it.verified else 1
            mark = "verified" if it.verified else "NOT FOUND IN TRANSCRIPT"
            lines.append(f"- **{getattr(it, label)}** [{mark}]")
            lines.append(f"  - evidence: \"{it.evidence_quote[:140]}\"")
    lines.append(f"\n*Evidence check: {checked - failed}/{checked} quotes verified against the source.*")
    return "\n".join(lines)

OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
(OUTD / "meridian_earnings_memo.md").write_text(memo_from(EarningsAnalysis.model_validate(analysis), "dry-run"))
print("outputs/meridian_earnings_memo.md written (dry-run).\n")

if HAS_KEY:
    live = verify_evidence(analyze(transcript, dry_run=False), transcript)
    typed_live = EarningsAnalysis.model_validate(live)      # the shape gate, before anything travels
    (OUTD / "meridian_earnings_memo.md").write_text(memo_from(typed_live, "LIVE"))
    ok = sum(1 for s in (typed_live.key_themes, typed_live.risks, typed_live.red_flags) for i in s if i.verified)
    n = sum(len(s) for s in (typed_live.key_themes, typed_live.risks, typed_live.red_flags))
    print(f"LIVE run: {ok}/{n} quotes verified. Memo overwritten with the live version.\n")
    llm.show(memo_from(typed_live, "LIVE"), title="Your memo, every claim marked")
else:
    print("No API key - the dry-run memo still demonstrates the whole pipeline.")

outputs/meridian_earnings_memo.md written (dry-run).

LIVE run: 12/13 quotes verified. Memo overwritten with the live version.



**Your memo, every claim marked**

# Meridian Semiconductor - Q2 FY2026 note (draft, from LIVE)

**Sentiment:** cautiously_positive - Management delivered strong headline results (record revenue, backlog, and operating margin) with confident framing around demand and capacity execution, and analysts largely validated the growth story. However, several analysts pressed hard on real vulnerabilities — recurring margin pressure mislabeled as one-time, rising customer concentration, DSO deterioration, and a significant unquantified regulatory risk excluded from guidance — and management's answers, while largely direct and transparent about exclusions, did not fully resolve the underlying tensions (e.g., CEO/CFO margin messaging, exact ramp cost trajectory). This mix of strong fundamentals with acknowledged unresolved risks supports a cautiously positive rather than unambiguously positive sentiment.

## Key themes

- **AI-driven data center demand significantly outpacing supply** [verified]
  - evidence: "AI demand is effectively infinite over any horizon we can plan for. The constraint is supply, and there we are executing well."
- **Data center segment now dominates revenue mix and is growing fast** [verified]
  - evidence: "Data center revenue was 1.72 billion dollars this quarter, up 61 percent year over year, and now represents 71 percent of total revenue."
- **Aggressive capacity investment to meet future demand** [verified]
  - evidence: "Our capacity agreement with our lead foundry partner, announced in May, secures a doubling of advanced-node wafer allocation through calenda"
- **Rising customer concentration risk amid hyperscaler wins** [verified]
  - evidence: "our largest customer represented 22 percent of revenue in the quarter, up from 15 percent a year ago."
- **Gross margin pressure from product ramp costs, characterized as one-time despite recurring pattern** [verified]
  - evidence: "this is the third consecutive quarter you've described ramp costs as one-time. At what point do we accept that ramp costs are simply a recur"
## Risks

- **Customer concentration with top customer at 22% of revenue and top three at ~41%, with one large customer having an internal silicon program that could in-source** [NOT FOUND IN TRANSCRIPT]
  - evidence: "The top three were approximately 41 percent of revenue this quarter... The customer you're referring to has had an internal program for year"
- **Pending export license review with Department of Commerce covering ~14% of trailing revenue, excluded from guidance with uncertain outcome and timing** [verified]
  - evidence: "the range of outcomes is too wide to model responsibly — from full licenses granted to a subset denied."
- **Rising DSO reflecting structural shift toward longer payment terms from hyperscaler customers, indicating potential working capital strain** [verified]
  - evidence: "I'd model DSO in the mid-60s going forward rather than a return to the 50s."
- **Large inventory build (up 41% YoY) concentrated in long-lead materials ahead of Corvus-2 ramp, creating execution and obsolescence risk if ramp falters** [verified]
  - evidence: "Inventory ended the quarter at 1.98 billion dollars, up 41 percent year over year, reflecting deliberate wafer and substrate purchases ahead"
- **Net debt position with convertible notes issued to fund capacity prepayments while continuing to increase capex guidance** [verified]
  - evidence: "We ended the quarter with 650 million dollars of cash and equivalents against 1.5 billion dollars of convertible notes issued in January to "
## Red flags

- **Repeated characterization of recurring ramp-related gross margin costs as 'one-time' for three consecutive quarters** [verified]
  - evidence: "You've now described ramp costs as one-time in each of the last three calls — this is the third consecutive quarter you've described ramp co"
- **Apparent tension between CEO's claim of structurally improving margins and CFO's refusal to guide a margin recovery timeline** [verified]
  - evidence: "how do we square that with the CFO declining to guide a recovery?"
- **Guidance explicitly excludes a known, material regulatory risk (export license review) affecting 14% of trailing revenue** [verified]
  - evidence: "I want to be explicit that our guidance excludes any impact from the export license review announced in June, which remains pending with the"

*Evidence check: 12/13 quotes verified against the source.*

## Deliverable checklist

- [ ] All three defects repaired: cause explained by Claude, fix accepted by the check
- [ ] Your `verify_evidence` catches **exactly 1** fabricated quote in the demonstration analysis — and you confirmed by searching the transcript that the quote is not there
- [ ] `outputs/meridian_earnings_memo.md` generated (live if you have a key) and committed to your repo
- [ ] You can say the session's rule in one sentence: code that runs is not code you can trust — checks are what convert one into the other

**Next:** `04-workflows-edgar.ipynb`, where the by-hand steps of these sessions become one automated workflow over live filings.